# 05 - RAG: knowledge base del campus

Nei notebook precedenti abbiamo visto tool, routing e memoria. Qui separiamo un concetto che spesso viene confuso con questi: **RAG**, cioe Retrieval-Augmented Generation.

L'idea e semplice: prima cerchiamo nei documenti rilevanti, poi chiediamo al modello di rispondere usando solo quel contesto.

## Obiettivi
- distinguere memoria, tool operativi e RAG
- trasformare documenti del lab in chunk recuperabili
- salvare chunk e vettori in un piccolo database SQLite
- recuperare i chunk piu pertinenti per una domanda
- generare una risposta con fonti
- esporre la retrieval come tool dentro un mini agente LangGraph


## Setup

Usiamo lo stesso helper degli altri notebook per creare il modello chat. La parte di retrieval sotto non richiede un modello di embedding esterno: per restare trasparenti useremo un embedding didattico basato su hashing.

Questo embedding non e semantico come un modello reale, ma rende visibili le parti importanti del RAG: chunk, vettori, top-k, score e fonti.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "lab_agentic").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lab_agentic.utils import get_chat_model

PROVIDER = "ollama_cloud"        # prova anche: "ollama"
MODEL_NAME = None

llm = get_chat_model(provider=PROVIDER, model=MODEL_NAME, temperature=0)
llm


## Prima: risposta senza knowledge base

Facciamo una domanda che ha una risposta precisa nei dati del lab. Senza documenti, il modello puo rispondere bene per caso, essere vago oppure inventare dettagli.


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

question = "Secondo le policy del lab, quanto vale una consegna dopo 4 giorni di ritardo?"

baseline = llm.invoke([
    SystemMessage(content="Sei un assistente per studenti universitari. Rispondi in italiano."),
    HumanMessage(content=question),
])

print(baseline.content)


La risposta sopra non e necessariamente sbagliata, ma non e **tracciabile**. Non sappiamo da quale fonte arrivi. Il RAG serve proprio a mettere una fase di retrieval esplicita prima della generazione.


## Carica una piccola knowledge base

Costruiamo documenti testuali partendo dai file gia presenti nel lab: FAQ, policy, catalogo corsi, eventi e aule.


In [ ]:
import csv
import json
from dataclasses import dataclass

DATA_DIR = PROJECT_ROOT / "data"

@dataclass
class Document:
    doc_id: str
    source: str
    title: str
    text: str


def first_markdown_heading(text: str, fallback: str) -> str:
    for line in text.splitlines():
        if line.startswith("#"):
            return line.lstrip("# ").strip()
    return fallback


def load_documents() -> list[Document]:
    documents: list[Document] = []

    for filename in ["faq.md", "policies.md"]:
        text = (DATA_DIR / filename).read_text(encoding="utf-8")
        documents.append(Document(
            doc_id=filename,
            source=filename,
            title=first_markdown_heading(text, filename),
            text=text,
        ))

    courses = json.loads((DATA_DIR / "course_catalog.json").read_text(encoding="utf-8"))
    for course in courses:
        code = course["code"]
        text = "\n".join([
            f"Corso {course['code']}: {course['title']}",
            f"Docente: {course['instructor']}",
            f"Orario: {', '.join(course['meeting_times'])}",
            f"Aula: {course['room']}",
            f"Parole chiave: {', '.join(course['keywords'])}",
            f"Descrizione: {course['description']}",
        ])
        documents.append(Document(
            doc_id=f"course:{code}",
            source=f"course_catalog.json#{code}",
            title=f"{code} - {course['title']}",
            text=text,
        ))

    events = json.loads((DATA_DIR / "events.json").read_text(encoding="utf-8"))
    for event in events:
        slug = event["name"].lower().replace(" ", "-")[:40]
        text = "\n".join([
            f"Evento: {event['name']}",
            f"Data: {event['date']}",
            f"Ora: {event['time']}",
            f"Luogo: {event['location']}",
            f"Tag: {', '.join(event['tags'])}",
        ])
        documents.append(Document(
            doc_id=f"event:{slug}",
            source=f"events.json#{event['name']}",
            title=event["name"],
            text=text,
        ))

    with open(DATA_DIR / "rooms.csv", newline="", encoding="utf-8") as f:
        for room in csv.DictReader(f):
            text = "\n".join([
                f"Aula: {room['room']}",
                f"Capienza: {room['capacity']}",
                f"Dotazione: {room['equipment']}",
            ])
            documents.append(Document(
                doc_id=f"room:{room['room']}",
                source=f"rooms.csv#{room['room']}",
                title=f"Aula {room['room']}",
                text=text,
            ))

    return documents


documents = load_documents()
print(f"Documenti caricati: {len(documents)}")
for doc in documents[:5]:
    print(f"- {doc.source}: {doc.title}")


## Chunking

Un documento intero puo essere troppo lungo o contenere parti non pertinenti. Lo dividiamo in chunk piccoli, con un po' di overlap per non tagliare il contesto in modo troppo netto.


In [ ]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    source: str
    title: str
    text: str


def chunk_text(text: str, max_words: int = 90, overlap: int = 20) -> list[str]:
    words = text.split()
    if len(words) <= max_words:
        return [text]

    chunks: list[str] = []
    step = max_words - overlap
    for start in range(0, len(words), step):
        part = words[start:start + max_words]
        if not part:
            break
        chunks.append(" ".join(part))
        if start + max_words >= len(words):
            break
    return chunks


def build_chunks(documents: list[Document]) -> list[Chunk]:
    chunks: list[Chunk] = []
    for doc in documents:
        for i, text in enumerate(chunk_text(doc.text)):
            chunks.append(Chunk(
                chunk_id=f"{doc.doc_id}::chunk-{i}",
                doc_id=doc.doc_id,
                source=doc.source,
                title=doc.title,
                text=text,
            ))
    return chunks


chunks = build_chunks(documents)
print(f"Chunk creati: {len(chunks)}")
print("\nEsempio chunk:\n")
print(chunks[0])


## Embedding didattico

Un sistema RAG reale usa un modello di embedding. Qui usiamo un embedding a hashing: ogni token finisce sempre nella stessa dimensione del vettore. E' meno potente, ma non richiede dipendenze extra e possiamo capirlo in pochi minuti.


In [ ]:
import hashlib
import math
import re

TOKEN_RE = re.compile(r"[A-Za-zÀ-ÿ0-9]+")


def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(text.lower())


class HashingEmbeddingModel:
    def __init__(self, dimensions: int = 256):
        self.dimensions = dimensions

    def embed(self, text: str) -> list[float]:
        vector = [0.0] * self.dimensions
        for token in tokenize(text):
            digest = hashlib.blake2b(token.encode("utf-8"), digest_size=8).digest()
            raw = int.from_bytes(digest, "big")
            index = raw % self.dimensions
            sign = 1.0 if (raw >> 63) == 0 else -1.0
            vector[index] += sign

        norm = math.sqrt(sum(value * value for value in vector))
        if norm == 0:
            return vector
        return [value / norm for value in vector]


embedder = HashingEmbeddingModel(dimensions=256)
example_vector = embedder.embed("consegne in ritardo e approvazione")

print(f"Dimensioni vettore: {len(example_vector)}")
print(f"Prime 12 dimensioni: {[round(x, 3) for x in example_vector[:12]]}")


## Salva chunk e vettori in SQLite

Questo e il nostro mini vector database. Ogni riga contiene testo, metadati e vettore serializzato in JSON.


In [ ]:
import sqlite3

DB_PATH = PROJECT_ROOT / "data" / "rag_campus.sqlite3"


def connect_db(db_path: Path = DB_PATH) -> sqlite3.Connection:
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    return conn


def build_vector_store(chunks: list[Chunk], embedder: HashingEmbeddingModel, db_path: Path = DB_PATH) -> None:
    with connect_db(db_path) as conn:
        conn.execute("DROP TABLE IF EXISTS chunks")
        conn.execute("""
            CREATE TABLE chunks (
                chunk_id TEXT PRIMARY KEY,
                doc_id TEXT NOT NULL,
                source TEXT NOT NULL,
                title TEXT NOT NULL,
                text TEXT NOT NULL,
                vector TEXT NOT NULL
            )
        """)

        rows = []
        for chunk in chunks:
            rows.append((
                chunk.chunk_id,
                chunk.doc_id,
                chunk.source,
                chunk.title,
                chunk.text,
                json.dumps(embedder.embed(chunk.text)),
            ))

        conn.executemany(
            "INSERT INTO chunks (chunk_id, doc_id, source, title, text, vector) VALUES (?, ?, ?, ?, ?, ?)",
            rows,
        )


build_vector_store(chunks, embedder)

with connect_db() as conn:
    count = conn.execute("SELECT COUNT(*) AS n FROM chunks").fetchone()["n"]
    row = conn.execute("SELECT source, text, vector FROM chunks LIMIT 1").fetchone()

print(f"DB creato: {DB_PATH}")
print(f"Righe indicizzate: {count}")
print(f"Fonte prima riga: {row['source']}")
print(f"Vettore serializzato: {row['vector'][:90]}...")


## Retrieval top-k

Ora possiamo trasformare la domanda in un vettore, confrontarla con i chunk nel database e ordinare per similarita coseno.


In [ ]:
def cosine_similarity(left: list[float], right: list[float]) -> float:
    return sum(a * b for a, b in zip(left, right))


def retrieve(query: str, k: int = 4, source_contains: str | None = None) -> list[dict]:
    query_vector = embedder.embed(query)

    with connect_db() as conn:
        rows = conn.execute(
            "SELECT chunk_id, doc_id, source, title, text, vector FROM chunks"
        ).fetchall()

    scored: list[dict] = []
    for row in rows:
        item = dict(row)
        if source_contains and source_contains not in item["source"]:
            continue
        item["score"] = cosine_similarity(query_vector, json.loads(item.pop("vector")))
        scored.append(item)

    return sorted(scored, key=lambda item: item["score"], reverse=True)[:k]


def print_retrieval_results(results: list[dict]) -> None:
    for i, item in enumerate(results, start=1):
        print(f"#{i} score={item['score']:.3f} source={item['source']}")
        print(item["text"][:500])
        print()


results = retrieve("consegna in ritardo dopo quattro giorni", k=4)
print_retrieval_results(results)


Questo e il punto centrale: prima ancora di chiamare il modello, possiamo controllare se il sistema ha recuperato pezzi utili. Se qui il contesto e sbagliato, la generazione finale sara fragile.


## Generazione grounded con fonti

Adesso passiamo al modello solo i chunk recuperati. Il prompt impone tre regole: usare il contesto, citare le fonti, ammettere quando il contesto non basta.


In [ ]:
RAG_SYSTEM_PROMPT = """
Sei un assistente RAG per il lab Agentic AI.
Regole:
- Rispondi solo usando il contesto recuperato.
- Se il contesto non contiene la risposta, dillo chiaramente.
- Cita le fonti tra parentesi quadre, per esempio [policies.md] o [course_catalog.json#NLP250].
- Non inventare date, percentuali, docenti, aule o policy.
""".strip()


def format_context(results: list[dict]) -> str:
    blocks = []
    for i, item in enumerate(results, start=1):
        blocks.append(
            f"[chunk {i}] fonte: {item['source']} | titolo: {item['title']} | score: {item['score']:.3f}\n"
            f"{item['text']}"
        )
    return "\n\n".join(blocks)


def answer_with_rag(question: str, k: int = 4) -> tuple[list[dict], str]:
    results = retrieve(question, k=k)
    context = format_context(results)
    response = llm.invoke([
        SystemMessage(content=RAG_SYSTEM_PROMPT),
        HumanMessage(content=f"Domanda: {question}\n\nContesto recuperato:\n{context}\n\nRispondi in italiano."),
    ])
    return results, response.content


rag_results, rag_answer = answer_with_rag(question, k=4)
print(rag_answer)


Ispezioniamo anche cosa e stato passato al modello. In un sistema RAG serio questa trace e fondamentale per debug, valutazione e fiducia.


In [ ]:
print_retrieval_results(rag_results)


## RAG non e la stessa cosa di un tool operativo

Un tool operativo incapsula una funzione precisa: ad esempio `policy_lookup("ritardo")` restituisce una risposta gia preparata. Il RAG invece cerca brani nei documenti e lascia al modello il compito di sintetizzare.

Entrambi sono utili, ma risolvono problemi diversi.


In [ ]:
from lab_agentic.utils import policy_lookup

print("TOOL OPERATIVO")
print(policy_lookup("consegna in ritardo"))
print()

print("RAG SEARCH")
print(format_context(retrieve("consegna in ritardo", k=2)))


## Esporre la retrieval come tool LangGraph

Ora trasformiamo la ricerca RAG in un tool. L'agente non modifica sistemi esterni: usa il tool solo per recuperare contesto dalla knowledge base.


In [ ]:
from langchain_core.tools import tool
from langgraph.graph import START, StateGraph
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition


@tool
def campus_rag_search(query: str, k: int = 4) -> str:
    """Cerca nella knowledge base del campus e restituisce chunk con fonte e score."""
    return format_context(retrieve(query, k=k))


rag_tools = [campus_rag_search]
llm_with_rag_tool = llm.bind_tools(rag_tools)

AGENT_SYSTEM_MESSAGE = SystemMessage(content="""
Sei un assistente per il lab Agentic AI.
Prima di rispondere a domande su corsi, policy, aule, eventi o FAQ, usa campus_rag_search.
Rispondi solo con informazioni supportate dal risultato del tool e cita le fonti.
Se il tool non basta, di' cosa manca.
""".strip())


def rag_assistant(state: MessagesState):
    return {"messages": [llm_with_rag_tool.invoke([AGENT_SYSTEM_MESSAGE] + state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("assistant", rag_assistant)
builder.add_node("tools", ToolNode(rag_tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

rag_graph = builder.compile()


In [ ]:
def ask_rag_agent(question: str):
    result = rag_graph.invoke({"messages": [HumanMessage(content=question)]})

    for message in result["messages"]:
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            print("AI tool calls:")
            for call in tool_calls:
                print(call)
            print()
        if getattr(message, "name", None) == "campus_rag_search":
            print("TOOL campus_rag_search output:")
            print(message.content[:900])
            print()

    print("RISPOSTA FINALE:")
    print(result["messages"][-1].content)
    return result


trace = ask_rag_agent("Quale corso mi consigli se voglio studiare chatbot, embedding e retrieval?")


## Mini-valutazione della retrieval

Prima di valutare la qualita della risposta finale, controlliamo se la retrieval porta nel contesto i fatti minimi necessari. Questa piccola valutazione non usa il modello: verifica fonti e parole attese nei chunk recuperati.


In [ ]:
EVAL_CASES = [
    {
        "question": "Quanto vale una consegna dopo quattro giorni di ritardo?",
        "expected_source": "policies.md",
        "must_contain": ["0", "proroga"],
    },
    {
        "question": "Quale corso parla di embedding e retrieval per chatbot?",
        "expected_source": "course_catalog.json#NLP250",
        "must_contain": ["NLP250", "embedding", "retrieval"],
    },
    {
        "question": "Mi serve una aula grande con registrazione e microfoni.",
        "expected_source": "rooms.csv#Aula-Magna",
        "must_contain": ["Aula-Magna", "120", "microfoni"],
    },
]


def evaluate_retrieval(cases: list[dict], k: int = 4) -> list[dict]:
    report = []
    for case in cases:
        results = retrieve(case["question"], k=k)
        joined = "\n".join(item["text"] for item in results).lower()
        sources = [item["source"] for item in results]
        report.append({
            "question": case["question"],
            "source_hit": case["expected_source"] in sources,
            "terms_hit": all(term.lower() in joined for term in case["must_contain"]),
            "top_sources": sources,
        })
    return report


for row in evaluate_retrieval(EVAL_CASES):
    status = "OK" if row["source_hit"] and row["terms_hit"] else "NO"
    print(f"[{status}] {row['question']}")
    print(f"  source_hit={row['source_hit']} terms_hit={row['terms_hit']}")
    print(f"  top_sources={row['top_sources']}")


## Sfida in aula

Modifica una sola cosa e misura l'effetto:

- dimensione dei chunk: `max_words`
- overlap tra chunk
- numero di risultati `k`
- prompt RAG
- domanda di valutazione
- filtro su una fonte specifica

Domanda guida: quando la risposta finale sbaglia, il problema nasce dalla retrieval, dal prompt o dal modello?
